# Unsupervised research plan and local NMF pilot

The first experiment uses the frozen document-level training cohort from Notebook 13. The researcher controls training in this notebook. The default execution verifies inputs, upstream artifacts and lexical coverage with the fit disabled. The corpus text, review decisions and learned document-level scores stay local.

This exploratory stage asks whether documents contain interpretable recurring language patterns about warming. It provides hypotheses and candidate evidence for the thesis on learned, culturally shared fear. The causal and interpretive claims require contextual reading, independent source evidence, and later measurement validation.

## Research questions and hypotheses

These exploratory questions connect to the thesis's longitudinal, measurement and event-centred research questions.

| Question | Hypothesis and expected lexical evidence | Competing explanation or weakening result |
| --- | --- | --- |
| **U1. Which recurring ways of talking about heat can we recover?** | Multiple interpretable frames recur across ranks and training resamples. Bodily exposure may group `too hot`, `air conditioning` and related coping terms. Disaster experience may group `bushfire`, `smoke` and `evacuation`. | Components track a dominant forum, sport, recipes, general weather or publisher templates. High-weight documents in the same component describe unrelated meanings. |
| **U2. Where and when do these patterns recur?** | One or more heat-risk or collective-future frames recur across unrelated release groups and multiple host types. Period differences persist on a common-support panel with a fixed host and format mix. | Pooled changes shrink or reverse on the matched panel, or depend on a few hosts or severe-event periods. Sparse common support can make the question unresolved. |
| **U3. Which excerpts distinguish immediate discomfort, shared risk narratives and reported or disputed fear?** | Some high-contribution documents connect heat to collective or intergenerational framing, disaster memory, public warnings, or attributed knowledge. Expressions such as `our children`, `future generations`, and `remember` are candidates for contextual review. | Negation, sarcasm, unattributed stock phrases, and quoting a third party can produce similar word weights. Counterexamples and ambiguous documents receive separate notes. Evidence of social learning requires independent support for transmission or shared narrative use. |

The examples guide close reading after fitting. The model receives the entire frozen vocabulary, with no seeded topic words or emotion labels. A single word can receive weight in several components. Negation scope, attribution, speaker identity and long phrases require the complete original document for interpretation.

## Pilot, stability checks and expected outcomes

1. **Pilot on training documents.** Start with 12 components and seed 1729 as a calibration choice. Use the frozen 20,000-parent cohort and retain the two all-zero feature rows in the audit while excluding them from the fit. The fitted factorization maps documents to weighted combinations of component vocabularies. The first local run produces component words, document-level relative contributions, representative parent IDs and a convergence receipt.
2. **Test stability before selecting a working rank.** Compare ranks 8, 12 and 16 across seeds 1729, 2211 and 4021. Align equal-rank components using cosine similarity and compare top-20 term overlap. Track components that split or merge across ranks. Repeat on group-resampled training data and check sensitivity to the largest host. Inspect high-contribution documents and counterexamples from multiple hosts before assigning names. Use reconstruction error as a numerical diagnostic alongside human interpretability. [NMF](https://www.nature.com/articles/44565) · [Human topic interpretation](https://papers.nips.cc/paper_files/paper/2009/hash/f92586a25bb3145facd64ab20fd554ff-Abstract.html) · [Factorization stability](https://www.sciencedirect.com/science/article/pii/S0957417417305948)
3. **Separate corpus composition and period effects.** Freeze vocabulary and components before projecting eligible documents. Restrict longitudinal comparisons to host and format strata observed in both periods, use a common mix, and report how many documents and hosts the comparison covers. Compare pooled and standardized estimates and show sensitivity to major hosts and independently documented event periods. Preserve unknown dates in the coverage table. Topic contributions are exploratory lexical measurements. [Topic models and document metadata](https://dtingley.scholars.harvard.edu/sites/g/files/omnuum8551/files/dtingley/files/topicmodelsopenendedexperiments.pdf)
4. **Record the outcome even if the strongest hypothesis weakens.** A stable cross-source pattern supports a shortlist for contextual annotation. A host- or event-dependent pattern supports a narrower conclusion about that source or event. An unstable fit yields a feature and corpus diagnostic, with definitions to revisit before substantive interpretation.

The calibration does not select a final rank or access validation and test text. Keep the test partition reserved until feature and interpretation rules are frozen.

## Operator controls

Run the input cells with the project kernel. To start an actual fit, set `RUN_LOCAL_NMF` to `True`, save the notebook, then execute the final cell locally. Change rank or seed deliberately for a new run. The code runs scikit-learn in the local Python process. [Solver parameters](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)

The same input and settings generate the same run identity in the same recorded environment. If a run exists, the cell compares component-aligned numeric outputs with its saved factors. Compare different ranks, seeds or library environments as separate experiments.

In [ ]:
# The researcher saves the notebook, then enables this flag for a local pilot fit.
RUN_LOCAL_NMF = False
N_COMPONENTS = 12
RANDOM_SEED = 1729
MAX_ITERATIONS = 250
SOLVER_TOLERANCE = 1e-4
BLAS_THREADS = 1


## Verify the exact development matrix

Check the frozen sample, body hashes, vocabulary indices and sparse vector invariants. Read the Parquet parts in both orders and confirm a single ordered CSR checksum. Scope is artifact verification and logical input replay.

In [ ]:
import hashlib
import json
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from scipy.sparse import csr_matrix

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/13_document_eda_and_unsupervised_preparation.ipynb").is_file()
)
if Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError("Select the registered project kernel before running this notebook.")

EDA = ROOT / ".cache/corpus-eda/document-eda-v2-source-use"
if not (EDA / "MANIFEST.json").is_file():
    raise FileNotFoundError("Run Notebooks 17 and 13 before starting the document experiment.")
EDA_MANIFEST = EDA / "MANIFEST.json"
manifest = json.loads(EDA_MANIFEST.read_text(encoding="utf-8"))
frozen_checks = manifest["feature_checks"]
if manifest["policy"]["fit_split"] != "train":
    raise ValueError("The pilot requires a training-only feature snapshot.")

def sha256_file(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


hold_pointer = json.loads(
    (ROOT / ".cache/corpus-freeze/SOURCE_USE_BODY_HOLDS.json").read_text(encoding="utf-8")
)
hold_manifest_path = ROOT / hold_pointer["freeze_manifest"]
if (
    hold_pointer["version"] != "source-use-native-body-hold-v1"
    or sha256_file(hold_manifest_path) != hold_pointer["freeze_manifest_sha256"]
    or manifest["identity"]["source_use_body_hold_manifest_sha256"]
    != hold_pointer["freeze_manifest_sha256"]
    or frozen_checks["canonical_eligible_training_frame"] != 641_180
):
    raise ValueError("The document features do not match the active source-use hold inventory.")
hold_manifest = json.loads(hold_manifest_path.read_text(encoding="utf-8"))
hold_files = hold_manifest["table_parts_sha256"]["source_use_body_holds"]
hold_parts = [hold_manifest_path.parent / name for name in hold_files]
if not hold_files or any(
    sha256_file(path) != digest
    for path, digest in zip(hold_parts, hold_files.values(), strict=True)
):
    raise ValueError("A source-use body hold file failed verification.")
held_release_rows = frozenset(pq.read_table(hold_parts, columns=["release_row"])["release_row"].to_pylist())
if len(held_release_rows) != 732:
    raise ValueError("The source-use hold roster did not reconcile.")


def canonical_sha256(value):
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(raw).hexdigest()


def verified_parts(root, entries, folder):
    entries = [
        item for item in entries
        if item["path"].startswith(f"{folder}/") and item["path"].endswith(".parquet")
    ]
    expected = {root / item["path"] for item in entries}
    observed = set((root / folder).glob("*.parquet"))
    if not expected or expected != observed:
        raise ValueError(f"The Parquet file roster changed for {folder}.")
    for item in entries:
        if sha256_file(root / item["path"]) != item["sha256"]:
            raise ValueError(f"A frozen {folder} part changed.")
    return sorted(expected)


feature_parts = verified_parts(EDA, manifest["files"], "train_sparse_tfidf")
cohort_parts = verified_parts(EDA, manifest["files"], "train_cohort")
vocabulary_path = EDA / "vocabulary.csv"
vocabulary_entry = next(item for item in manifest["files"] if item["path"] == "vocabulary.csv")
if sha256_file(vocabulary_path) != vocabulary_entry["sha256"]:
    raise ValueError("The frozen feature vocabulary changed.")
vocabulary = pd.read_csv(vocabulary_path, keep_default_na=False).sort_values("index")
feature_count = frozen_checks["vocabulary_size"]
if len(vocabulary) != feature_count or not np.array_equal(vocabulary["index"], np.arange(feature_count)):
    raise ValueError("The vocabulary does not contain one row per ordered feature.")
if not vocabulary["term"].is_unique:
    raise ValueError("The feature vocabulary repeats a term.")
cohort = pq.read_table(cohort_parts).sort_by("release_row")
if held_release_rows.intersection(cohort["release_row"].to_pylist()):
    raise ValueError("The exploratory cohort contains a source-use-held document.")

if len(cohort) != frozen_checks["rows"]:
    raise ValueError("The sampled document roster differs from its receipt.")

def load_matrix(parts):
    records = pq.read_table(parts).sort_by("release_row")
    release_rows = records["release_row"].to_numpy()
    if not np.array_equal(release_rows, cohort["release_row"].to_numpy()) or not (
        records["clean_sha256"].combine_chunks().equals(cohort["clean_sha256"].combine_chunks())
    ) or np.unique(release_rows).size != len(release_rows):
        raise ValueError("A matrix row does not match its ordered and hashed parent document.")
    vectors = records["tfidf"].combine_chunks()
    if (
        vectors.null_count
        or any(vectors.field(i).null_count for i in range(vectors.type.num_fields))
        or not np.all(vectors.field("type").to_numpy() == 0)
        or not np.all(vectors.field("size").to_numpy() == feature_count)
    ):
        raise ValueError("Expected one sparse Spark vector per document and a fixed dimension.")
    indices = vectors.field("indices")
    values = vectors.field("values")
    pointers = indices.offsets.to_numpy()
    indices_array = indices.flatten().to_numpy()
    values_array = values.flatten().to_numpy()
    if (
        not np.array_equal(pointers, values.offsets.to_numpy())
        or not np.array_equal(np.diff(pointers), records["nonzero_features"].to_numpy())
        or np.any(indices_array < 0)
        or np.any(indices_array >= feature_count)
        or not np.isfinite(values_array).all()
        or np.any(values_array <= 0.0)
    ):
        raise ValueError("The stored sparse vectors have invalid positions or weights.")
    matrix = csr_matrix(
        (values_array, indices_array, pointers), shape=(len(release_rows), feature_count)
    )
    if not matrix.has_canonical_format or matrix.nnz != frozen_checks["nonzero_values"]:
        raise ValueError("The matrix is missing values or has duplicate or unordered columns.")
    nonempty = np.diff(matrix.indptr) > 0
    if int((~nonempty).sum()) != frozen_checks["zero_feature_documents"]:
        raise ValueError("The zero-vector roster changed.")
    return matrix


def matrix_sha256(matrix, row_ids):
    signature = hashlib.sha256()
    for array, dtype in (
        (matrix.shape, "<i8"), (row_ids, "<i8"), (matrix.indptr, "<i8"),
        (matrix.indices, "<i8"), (matrix.data, "<f8"),
    ):
        signature.update(np.ascontiguousarray(array, dtype=dtype).tobytes())
    return signature.hexdigest()


X = load_matrix(feature_parts)
row_ids = cohort["release_row"].to_numpy()
input_sha = matrix_sha256(X, row_ids)
reverse_sha = matrix_sha256(load_matrix(list(reversed(feature_parts))), row_ids)
if reverse_sha != input_sha:
    raise ValueError("The logical feature matrix depends on input file order.")
fit_mask = np.diff(X.indptr) > 0
packages = {
    name: version(name)
    for name in ("scikit-learn", "numpy", "scipy", "pyarrow", "threadpoolctl")
}
display(pd.DataFrame({
    "Measure": ["Sampled parents", "Nonzero training rows", "Ordered features", "Nonzero values"],
    "Value": [len(row_ids), int(fit_mask.sum()), feature_count, X.nnz],
}))
print(f"Verified the same logical matrix in both file orders: {input_sha}")


Local input check covered 711 listed files across 10 manifests and 75 frozen QA packets and receipts. Reading the Parquet parts in both orders produced the same logical CSR SHA-256, `87d94fdd895343994b28bed9787ff5c444ab73f13534ca6e5ab756a0010b57ba`. The check used 20,000 parent rows and 24,000 features. The fitting cell remains unexecuted.

## Verify the saved preprocessing and chunk checkpoints

Read the earlier stage manifests, verify listed output files and their recorded row counts, and check the frozen program QA packets and decision files. The [local replay guide](REPRODUCIBILITY.md) covers the fixture, review replay, and full rebuild routes. This cell checks the existing checkpoint chain.

In [ ]:
preprocessing = ROOT / "data/processed/preprocessing"
structure = ROOT / "work_packages/formal_corpus_release_20261117"
review = ROOT / ".cache/chunk-policy-review"
reference_root = ROOT / ".cache/chunk-materialization/reference-v1"
refinement_root = ROOT / ".cache/chunk-refinement/paragraph-v2"
reference_manifest = preprocessing / "formal_layout_step_v1_handoff_v1_reference_chunks_v1/MANIFEST.json"
checkpoints = {
    "03 Structure": structure / "structure-v2/MANIFEST.json",
    "03 Overlap": structure / "structure-v2-overlap-v1/MANIFEST.json",
    "04 First pass": preprocessing / "formal_first_pass_v1/MANIFEST.json",
    "05 Document handoff": preprocessing / "formal_layout_step_v1_handoff_v1/MANIFEST.json",
    "06 Reference": reference_manifest,
    "08 Scope and fallback": preprocessing / "formal_layout_step_v1_handoff_v1_chunk_policy_candidate_v1/MANIFEST.json",
    "11 Program QA": review / "program-chunk-qa-v1/MANIFEST.json",
    "12 Reference materialization": reference_root / "MANIFEST.json",
    "13 Document EDA": EDA_MANIFEST,
    "14 Refinement proposals": refinement_root / "MANIFEST.json",
}


def data_file_entries(manifest_record):
    parts = list(manifest_record.get("files", [])) + list(manifest_record.get("records_files", []))
    for table in manifest_record.get("tables", {}).values():
        parts.extend(table["files"] if isinstance(table, dict) else table)
    return parts


checks = []
for label, path in checkpoints.items():
    record = json.loads(path.read_text(encoding="utf-8"))
    entries = data_file_entries(record)
    if not path.is_file() or not entries:
        raise ValueError(f"No verifiable local data files for {label}.")
    for item in entries:
        file = path.parent / item["path"]
        if sha256_file(file) != item["sha256"]:
            raise ValueError(f"A saved {label} asset changed: {file.name}.")
        if "rows" in item and pq.ParquetFile(file).metadata.num_rows != item["rows"]:
            raise ValueError(f"The saved row count changed in {label}: {file.name}.")
    checks.append((label, len(entries), sha256_file(path)))

qa_manifest = json.loads(checkpoints["11 Program QA"].read_text(encoding="utf-8"))
qa_root = checkpoints["11 Program QA"].parent
for label, receipt_sha in manifest["chunk_qa_receipts"].items():
    packet = qa_root / qa_manifest["qa_batches"][label]["file"]
    receipt = qa_root / "decisions" / f"{label}.jsonl"
    if (
        sha256_file(packet) != qa_manifest["qa_batches"][label]["sha256"]
        or sha256_file(receipt) != receipt_sha
    ):
        raise ValueError("A frozen QA packet or receipt changed after the document EDA handoff.")
if len(manifest["chunk_qa_receipts"]) != len(qa_manifest["qa_batches"]):
    raise ValueError("The frozen QA packet and receipt rosters did not reconcile.")

handoff_sha = sha256_file(checkpoints["05 Document handoff"])
if (
    manifest["identity"]["document_handoff_manifest_sha256"] != handoff_sha
    or manifest["identity"]["chunk_qa_manifest_sha256"] != sha256_file(checkpoints["11 Program QA"])
    or qa_manifest["reference_manifest_sha256"] != sha256_file(reference_manifest)
    or qa_manifest["full_review_closure_sha256"] != sha256_file(
        review / "full-nonparagraph-v1/FULL_REVIEW_CLOSURE.json"
    )
):
    raise ValueError("A document, reference or semantic replay binding changed.")
display(pd.DataFrame(checks, columns=["Checkpoint", "Files verified", "Manifest SHA-256"]))
print(f"Verified {len(manifest['chunk_qa_receipts'])} frozen program QA packets and receipts.")


## Check representation of the hypothesized vocabulary

The document frequencies below describe the existing TF–IDF vocabulary. Terms outside it can still appear in source documents. Use their absence from the feature space to prioritize context checks or a separately versioned feature experiment.

In [ ]:
watch_terms = {
    "Heat and bodily experience": ["heatwave", "too hot", "heat stress", "air conditioning"],
    "Disaster memory": ["bushfire", "smoke", "evacuation", "black summer", "remember"],
    "Collective and future framing": ["our children", "our kids", "future generations"],
    "Emotion and competing interpretations": [
        "fear", "afraid", "worried", "not", "not worried", "not afraid", "scientists say", "media hype"
    ],
}
lookups = pd.DataFrame(
    [(frame, term) for frame, terms in watch_terms.items() for term in terms],
    columns=["hypothesized_frame", "term"],
)
coverage = lookups.merge(
    vocabulary[["term", "document_frequency"]], on="term", how="left", validate="one_to_one"
)
coverage["in_frozen_vocabulary"] = coverage["document_frequency"].notna()
coverage["document_frequency"] = coverage["document_frequency"].astype("Int64")
display(coverage)


## Optional fit started by the researcher

When enabled, store the versioned local factors and the first 20 ranked terms and 10 document IDs per component. Inspect fit warnings and read representative texts and counterexamples before assigning component names. The aggregate and period comparisons follow the stability and source checks described above.

In [ ]:
if not RUN_LOCAL_NMF:
    print("Inputs verified. The local NMF cell is disabled.")
else:
    import platform

    import nbformat
    import tempfile
    import warnings

    from scipy.optimize import linear_sum_assignment
    from sklearn.decomposition import NMF
    from sklearn.exceptions import ConvergenceWarning
    from threadpoolctl import threadpool_info, threadpool_limits

    model_parameters = {
        "n_components": N_COMPONENTS,
        "init": "nndsvdar",
        "solver": "cd",
        "beta_loss": "frobenius",
        "max_iter": MAX_ITERATIONS,
        "tol": SOLVER_TOLERANCE,
        "random_state": RANDOM_SEED,
        "shuffle": False,
        "alpha_W": 0.0,
        "alpha_H": 0.0,
    }
    with threadpool_limits(limits=BLAS_THREADS):
        runtime_threads = sorted(
            (
                {key: library.get(key) for key in
                 ("internal_api", "version", "num_threads", "architecture")}
                for library in threadpool_info()
            ),
            key=canonical_sha256,
        )
        model = NMF(**model_parameters)
        with warnings.catch_warnings(record=True) as observed_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            doc_matrix = model.fit_transform(X[fit_mask])
        topics = model.components_

    saved_notebook = nbformat.read(
        ROOT / "notebooks/15_unsupervised_research_plan_and_local_pilot.ipynb", as_version=4
    )
    fit_code = next(cell.source for cell in saved_notebook.cells if cell.id == "15-fit")
    run_identity = {
        "protocol": "document_nmf_local_pilot_v1",
        "eda_manifest_sha256": sha256_file(EDA_MANIFEST),
        "vocabulary_sha256": vocabulary_entry["sha256"],
        "csr_sha256": input_sha,
        "uv_lock_sha256": sha256_file(ROOT / "uv.lock"),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "packages": packages,
        "linear_algebra_runtime": runtime_threads,
        "blas_threads": BLAS_THREADS,
        "model_parameters": model_parameters,
        "notebook_fit_source_sha256": hashlib.sha256(fit_code.encode("utf-8")).hexdigest(),
    }
    if (
        np.any(topics < 0) or np.any(doc_matrix < 0)
        or not np.isfinite(topics).all() or not np.isfinite(doc_matrix).all()
    ):
        raise ValueError("The factorization returned invalid weights.")
    component_scale = topics.sum(axis=1)
    if np.any(component_scale <= 0):
        raise ValueError("At least one component is empty. Keep this configuration for diagnosis.")
    normalized_topics = topics / component_scale[:, None]
    contribution = doc_matrix * component_scale[None, :]
    totals = contribution.sum(axis=1, keepdims=True)
    scores = np.divide(contribution, totals, out=np.zeros_like(contribution), where=totals > 0)

    tokens = vocabulary["term"].to_numpy()
    fitted_rows = row_ids[fit_mask]
    labels = [f"component_{i:02d}" for i in range(N_COMPONENTS)]
    top_terms = []
    exemplars = []
    for i, label in enumerate(labels):
        term_order = np.lexsort((np.arange(len(tokens)), -normalized_topics[i]))[:20]
        top_terms.extend(
            (label, rank, tokens[j], normalized_topics[i, j])
            for rank, j in enumerate(term_order, 1)
        )
        document_order = np.lexsort((fitted_rows, -scores[:, i]))[:10]
        exemplars.extend(
            (label, rank, int(fitted_rows[j]), float(scores[j, i]))
            for rank, j in enumerate(document_order, 1)
        )
    terms_table = pd.DataFrame(
        top_terms, columns=["component", "rank", "term", "normalized_term_weight"]
    )
    exemplars_table = pd.DataFrame(
        exemplars, columns=["component", "rank", "release_row", "relative_topic_contribution"]
    )
    fit_metrics = {
        "training_documents": int(fit_mask.sum()),
        "feature_zero_documents": int((~fit_mask).sum()),
        "all_zero_component_score_documents": int(np.count_nonzero(totals == 0)),
        "iterations": int(model.n_iter_),
        "reconstruction_error": float(model.reconstruction_err_),
        "convergence_warning": any(
            issubclass(item.category, ConvergenceWarning) for item in observed_warnings
        ),
    }
    run_id = canonical_sha256(run_identity)[:20]
    work = ROOT / ".cache/unsupervised-nmf"
    work.mkdir(mode=0o700, exist_ok=True)
    run_dir = work / run_id
    if run_dir.exists():
        prior = json.loads((run_dir / "MANIFEST.json").read_text(encoding="utf-8"))
        if prior["identity"] != run_identity:
            raise ValueError("The local run directory already belongs to different inputs.")
        for item in prior["files"]:
            if sha256_file(run_dir / item["path"]) != item["sha256"]:
                raise ValueError("A saved NMF artifact failed its checksum.")
        for key in ("iterations", "convergence_warning", "all_zero_component_score_documents"):
            if prior["metrics"][key] != fit_metrics[key]:
                raise ValueError(f"The saved fit diagnostic changed for {key}.")
        np.testing.assert_allclose(
            fit_metrics["reconstruction_error"],
            prior["metrics"]["reconstruction_error"],
            rtol=1e-8, atol=1e-10,
        )
        with np.load(run_dir / "factors.npz", allow_pickle=False) as saved:
            np.testing.assert_array_equal(saved["release_rows"], fitted_rows)
            prev_topics = saved["topic_term_weights"]
            prev_docs = saved["document_contributions"]
            denominator = np.linalg.norm(prev_topics, axis=1)[:, None] * np.linalg.norm(
                normalized_topics, axis=1
            )[None, :]
            similarity = (prev_topics @ normalized_topics.T) / denominator
            rows, cols = linear_sum_assignment(-similarity)
            if not np.array_equal(rows, np.arange(N_COMPONENTS)):
                raise ValueError("Component alignment did not cover the previously saved model.")
            np.testing.assert_allclose(prev_topics, normalized_topics[cols], rtol=1e-8, atol=1e-10)
            np.testing.assert_allclose(prev_docs, scores[:, cols], rtol=1e-8, atol=1e-10)
        print(f"Local replay matched the saved factors for {run_id}.")
    else:
        with tempfile.TemporaryDirectory(prefix=".nmf-", dir=work) as tmp:
            stage = Path(tmp) / "snapshot"
            stage.mkdir(mode=0o700)
            np.savez_compressed(
                stage / "factors.npz",
                release_rows=fitted_rows,
                topic_term_weights=normalized_topics,
                document_contributions=scores,
            )
            terms_table.to_csv(stage / "top_terms.csv", index=False)
            exemplars_table.to_csv(stage / "representative_documents.csv", index=False)
            result = {
                "identity": run_identity,
                "metrics": fit_metrics,
                "files": [
                    {"path": file.name, "sha256": sha256_file(file)}
                    for file in sorted(stage.iterdir())
                ],
            }
            (stage / "MANIFEST.json").write_text(
                json.dumps(result, indent=2, sort_keys=True) + "\n", encoding="utf-8"
            )
            if run_dir.exists():
                raise FileExistsError("A local operator already published this fit.")
            stage.rename(run_dir)
        print(f"Saved local factors at {run_dir.relative_to(ROOT)}.")
    display(pd.DataFrame(fit_metrics.items(), columns=["Measure", "Value"]))
    display(terms_table.where(terms_table["rank"] <= 8).dropna())
